# UK Train Rides: Load the Tables into SQL Server

This notebook loads the 11 CSV files from `data/processed/` into the `UK_Train_Rides` database in SQL Server.

Before running it, the database and the empty tables must exist. They are created by two scripts in the `sql` folder:

1. `01_create_database.sql`
2. `02_create_tables.sql`

The tables already have their primary keys, foreign keys and check constraints, so SQL Server will reject any row that breaks them.

## 1. Connect to SQL Server

We use SQLAlchemy with the `pyodbc` driver. The connection uses Windows Authentication, so no username or password is needed.

`SERVER` is the name shown in the SSMS connection window. Change it if you run the notebook on another computer.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

SERVER = "LAPTOP-Q470H8LN"
DATABASE = "UK_Train_Rides"
DRIVER = "ODBC Driver 18 for SQL Server"

connection_url = (
    f"mssql+pyodbc://@{SERVER}/{DATABASE}"
    f"?driver={DRIVER.replace(' ', '+')}"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)

# fast_executemany sends the rows in batches, which makes loading much faster
engine = create_engine(connection_url, fast_executemany=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT DB_NAME()")).scalar())

UK_Train_Rides


The connection works and we are connected to the `UK_Train_Rides` database.

## 2. Read the CSV files

The tables are listed in the order they must be loaded. A table with a foreign key can only be loaded after the table it points to:

- The dimension tables come first. `Dim_Route` comes after `Dim_Station` because it points to it.
- `Fact_Journey` comes next, because it points to route, date and delay reason.
- `Fact_TicketSales` comes last, because it points to `Fact_Journey`.

Two reading options are important:

- By default pandas turns some words into empty values, and `None` is one of them. That would break `Dim_Railcard`, where `None` is a real value meaning "no railcard". With `keep_default_na=False` and `na_values=[""]`, only truly empty cells become empty values.
- `Delay_Minutes` is read as a nullable integer, so the empty values of cancelled journeys stay empty.

In [2]:
load_order = [
    "Dim_Station",
    "Dim_Route",
    "Dim_Date",
    "Dim_TicketType",
    "Dim_TicketClass",
    "Dim_Railcard",
    "Dim_PaymentMethod",
    "Dim_PurchaseType",
    "Dim_DelayReason",
    "Fact_Journey",
    "Fact_TicketSales",
]

tables = {}
for name in load_order:
    tables[name] = pd.read_csv(f"../data/processed/{name}.csv",
                               keep_default_na=False, na_values=[""],
                               dtype={"Delay_Minutes": "Int64"})
    print(f"{name:<18} {len(tables[name]):>6} rows")

Dim_Station            32 rows
Dim_Route              65 rows
Dim_Date              152 rows
Dim_TicketType          3 rows
Dim_TicketClass         2 rows
Dim_Railcard            4 rows
Dim_PaymentMethod       3 rows
Dim_PurchaseType        2 rows
Dim_DelayReason         6 rows
Fact_Journey        19871 rows
Fact_TicketSales    31653 rows


## 3. Empty the tables

To make the notebook safe to run more than once, we first delete any rows already in the tables. We delete in the reverse order, starting with the fact tables, because SQL Server does not allow deleting a row that another table still points to.

In [3]:
with engine.begin() as conn:
    for name in reversed(load_order):
        conn.execute(text(f"DELETE FROM {name}"))

print("All tables are empty")

All tables are empty


## 4. Load the data

`to_sql` with `if_exists="append"` inserts the rows into the existing tables. It does not create new tables, so the data types and constraints from `02_create_tables.sql` are kept.

In [4]:
for name in load_order:
    tables[name].to_sql(name, engine, if_exists="append", index=False, chunksize=1000)
    print(f"Loaded {name}")

Loaded Dim_Station
Loaded Dim_Route
Loaded Dim_Date
Loaded Dim_TicketType
Loaded Dim_TicketClass
Loaded Dim_Railcard
Loaded Dim_PaymentMethod
Loaded Dim_PurchaseType
Loaded Dim_DelayReason
Loaded Fact_Journey
Loaded Fact_TicketSales


All 11 tables were loaded without errors. If any row had broken a primary key, foreign key or check constraint, SQL Server would have stopped the load with an error.

## 5. Check the loaded data

### 5.1 Row counts

We count the rows of each table in SQL Server and compare them with the CSV files.

In [5]:
rows = []
with engine.connect() as conn:
    for name in load_order:
        sql_count = conn.execute(text(f"SELECT COUNT(*) FROM {name}")).scalar()
        rows.append((name, len(tables[name]), sql_count, len(tables[name]) == sql_count))

pd.DataFrame(rows, columns=["Table", "CSV rows", "SQL rows", "Match"])

,Table,CSV rows,SQL rows,Match
0,Dim_Station,32,32,True
1,Dim_Route,65,65,True
2,Dim_Date,152,152,True
3,Dim_TicketType,3,3,True
4,Dim_TicketClass,2,2,True
5,Dim_Railcard,4,4,True
6,Dim_PaymentMethod,3,3,True
7,Dim_PurchaseType,2,2,True
8,Dim_DelayReason,6,6,True
9,Fact_Journey,19871,19871,True


Every table has the same number of rows in SQL Server as in its CSV file.

### 5.2 Total price

The total price in SQL Server must be the same as in the raw data (£741,921).

In [6]:
with engine.connect() as conn:
    sql_total = conn.execute(text("SELECT SUM(Price) FROM Fact_TicketSales")).scalar()

print("Total price in SQL Server: £", sql_total)
assert float(sql_total) == 741921

Total price in SQL Server: £ 741921.00


### 5.3 Journey status

A quick query on `Fact_Journey` to see the journey status counted once per journey.

In [7]:
query = '''
SELECT Journey_Status, COUNT(*) AS Journeys
FROM Fact_Journey
GROUP BY Journey_Status
ORDER BY Journeys DESC
'''
pd.read_sql(text(query), engine)

,Journey_Status,Journeys
0,On Time,18019
1,Delayed,1062
2,Cancelled,790


The result is the same as in the cleaning notebook: 18,019 on time, 1,062 delayed and 790 cancelled journeys.

### 5.4 The constraints are working

Finally, we try to insert two wrong tickets on purpose, to show that the database protects the data by itself:

1. A ticket for a journey that does not exist (`JourneyID` 999999). The foreign key should reject it.
2. A ticket with a negative price. The check constraint should reject it.

In [8]:
import re

bad_rows = {
    "Ticket for a journey that does not exist":
        "INSERT INTO Fact_TicketSales VALUES "
        "('TEST-1', 999999, 20240101, 1, 1, 1, 1, 1, '10:00:00', 10, 'No', 0)",
    "Ticket with a negative price":
        "INSERT INTO Fact_TicketSales VALUES "
        "('TEST-2', 1, 20240101, 1, 1, 1, 1, 1, '10:00:00', -5, 'No', 0)",
}

for test, sql in bad_rows.items():
    try:
        with engine.begin() as conn:
            conn.execute(text(sql))
        print(f"{test}: ACCEPTED (this should not happen)")
    except Exception as error:
        reason = re.search(r'conflicted with the (.+? constraint "[^"]+")', str(error))
        print(f"{test}: REJECTED by the {reason.group(1) if reason else 'database'}")

Ticket for a journey that does not exist: REJECTED by the FOREIGN KEY constraint "FK_Ticket_Journey"
Ticket with a negative price: REJECTED by the CHECK constraint "CK_Ticket_Price"


Both wrong tickets were rejected, and SQL Server named the constraint that stopped each one. No test rows were added to the table.

## Summary

- The 11 tables were loaded into SQL Server in the correct order: dimensions first, then `Fact_Journey`, then `Fact_TicketSales`.
- Row counts in SQL Server match the CSV files, and the total price is still £741,921.
- The database rejects rows that break a foreign key or a check constraint.

The database is ready to be connected to Power BI.